# Infinaxis System Setup
This notebook goes through how to set up the needed Axis for 3 multiaxis system:
- A 4 axis mandrel type printer with a bed rotating around X (type A)
- A 4 axis tilting nozzle printer with the nozzle rotating aroung Y (type B)
- A 5 axis tilting and rotating bed printer with the bed rotating aroung X and Z (type A and C, respectively)

As for terminology infinaxis follows the convention that rotational axes are Type A, B or C, based on if the rotate around X, Y or Z, respectively. In the case of a stacked chain of rotating axis the type depends on the orientation when the system is at its origin position. Hence an A-C chain means that when the A axis is at 0 then the second rotating axis rotates around Z, and is hence type C.

Please note that infinaxis always expects that your system has the options to move in X,Y and Z outside the additional axis you configure. If your system cannot move in all 3 axes, infinaxis can still be used, it is just up to you to design the toolpaths so that the missing axis or axes are static.

The movement in XYZ does not need to be realised through a linear carriage system. We have succesfully used infinaxis to combine a 6-axis robot with a rotating table. In this case the resulting XYZ movement from infinaxis was then further post processed by the inverse kinematics of the 6 axis robot. The role of infinaxis was to relate the rotating table and the robot.

In [ ]:
import lab.fullcontrol.infinaxis as fci
from math import sin, cos, tau

# Common print settings
EW = 0.4
EH = 0.2

print_settings = {'extrusion_width': EW,'extrusion_height': EH}

#### Mandrel (4-axis)
This printer has one additional A axis on the bed chain.

In [ ]:
# The head chain is empty as no additional axis are present on the tool-head side of the printer.
head_chain = []

# The extra A axis is included in the bed chain.
# For a mandrel type printer an offset could be added to conform to the radius of the mandrel. 
# But as this type of printer is often used for symmetric deposition around the mandrel as opposed to a singular point on the surface,
# it can be more beneficial to include the mandrel offset in the toolpath design as opposed to during axes setup.
bed_chain = [fci.Axis(name='A')] # If the axis name is one of the three types its type is automatically inferred. If a custom name is needed the 'type' argument can be used, see the infinaxis_controls tutorial for more info.

#Once the head and bed chain are configured they are used to configure the custom point.
#This point can now be used in a similar manner to the normal fc.Point, but it will have the additional arguments based on the axis names.
#This point is hence fully configured by using Point(x,y,z,a).
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

# Simple cylinder demonstration path
r = 50 # radius of the spiral
density = 36 # number of points per revolution
layers = 60 # number of layers in the x direction
w = EW # spacing of each line

steps = []
steps.append(fci.Printer(print_speed=2160))
for i in range(layers):
    for j in range(density):
        angle = 360*(i+j/density)
        steps.append(Point(x=((i+j/density)*w),y=r*sin(angle/360*tau), z=r*cos(angle/360*tau), a=angle))

for step in steps:
    if isinstance(step, fci.Point):
        # color is a gradient from A=0 deg (blue) to A=360 (red)
        step.color = [((step.a%360)/360), 0, 1-((step.a%360)/360)]

fci.transform(steps, 'plot', fci.PlotControls(color_type='manual',style='tube', zoom=0.75), show_tips=False)
gcode = fci.transform(steps,'gcode',gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))